# Notebook 07 — Empaquetado y Despliegue en Sala de Control

## Sensores Virtuales y Analizadores en Línea

En este notebook final empaquetamos todos los artefactos de producción dentro de un único contrato versionado (`sensor_virtual_v1.pkl`):
- Modelo puntual **LightGBM + Optuna**
- Modelos cuantílicos **`q10` / `q90` + offset CQR** (banda de confianza 80%)
- Monitor multivariado **PCA (`Hotelling T²` y `SPE`)**
- Parámetros de **Filtro de Kalman** e histórico de evaluación
- Script completo de la aplicación HMI **Streamlit (`main.py`)** con explicabilidad **SHAP en vivo**.

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

WORK_DIR = "/content" if os.path.isdir("/content") else "."
df = pd.read_csv(os.path.join(WORK_DIR, "sensor_model_ready.csv"))
feature_cols = [c for c in df.columns if c not in ["date", "flash_point_lab_C", "split"]]

tr = df["split"] == "train"
va = df["split"] == "val"
te = df["split"] == "test"

X_tr, y_tr = df.loc[tr, feature_cols], df.loc[tr, "flash_point_lab_C"].values
X_va, y_va = df.loc[va, feature_cols], df.loc[va, "flash_point_lab_C"].values
X_te, y_te = df.loc[te, feature_cols], df.loc[te, "flash_point_lab_C"].values
X_hist = pd.concat([X_tr, X_va], axis=0)
y_hist = np.concatenate([y_tr, y_va])

# 1. Modelo puntual final entrenado hasta fin de Q3
model_point = lgb.LGBMRegressor(
    n_estimators=320, learning_rate=0.04, max_depth=5, num_leaves=20,
    min_child_samples=8, subsample=0.85, colsample_bytree=0.85,
    random_state=42, n_jobs=-1, verbose=-1
).fit(X_hist, y_hist)

# 2. Modelos cuantílicos P10-P90 + Calibración CQR
q_pars = dict(n_estimators=90, learning_rate=0.04, max_depth=3, num_leaves=8, min_child_samples=20, random_state=42, verbose=-1)
model_q10 = lgb.LGBMRegressor(objective="quantile", alpha=0.10, **q_pars).fit(X_tr, y_tr)
model_q90 = lgb.LGBMRegressor(objective="quantile", alpha=0.90, **q_pars).fit(X_tr, y_tr)
mid = len(X_va) // 2
scores_cal = np.maximum(model_q10.predict(X_va.iloc[:mid]) - y_va[:mid], y_va[:mid] - model_q90.predict(X_va.iloc[:mid]))
cqr_offset = float(np.quantile(scores_cal, 0.80 * (1.0 + 1.0 / mid)))

# Reajustar cuantiles con Train+Val manteniendo calibración
model_q10.fit(X_hist, y_hist)
model_q90.fit(X_hist, y_hist)

# 3. Monitor PCA (Hotelling T2 y SPE)
sc_pca = StandardScaler()
X_hist_s = sc_pca.fit_transform(X_hist)
pca = PCA(n_components=0.95, svd_solver="full").fit(X_hist_s)
s_hist = pca.transform(X_hist_s)
t2_hist = np.sum((s_hist ** 2) / pca.explained_variance_, axis=1)
spe_hist = np.sum((X_hist_s - pca.inverse_transform(s_hist)) ** 2, axis=1)

# 4. Construir histórico de evaluación Q4 para la pestaña de monitoreo de la app
p_stat = model_point.predict(X_te)
lo_te = model_q10.predict(X_te) - cqr_offset
hi_te = model_q90.predict(X_te) + cqr_offset
X_te_s = sc_pca.transform(X_te)
s_te = pca.transform(X_te_s)
t2_te = np.sum((s_te ** 2) / pca.explained_variance_, axis=1)
spe_te = np.sum((X_te_s - pca.inverse_transform(s_te)) ** 2, axis=1)

bias, P, Q, R = 0.0, 1.0, 0.25, 0.30
p_kal, biases = [], []
for yt, yp in zip(y_te, p_stat):
    p_kal.append(yp + bias); biases.append(bias)
    P_p = P + Q; K = P_p / (P_p + R); bias = bias + K * ((yt - yp) - bias); P = (1.0 - K) * P_p
p_kal = np.array(p_kal)

history_eval = pd.DataFrame({
    "date": df.loc[te, "date"].values,
    "y_true": y_te,
    "y_pred_static": p_stat,
    "y_pred_kalman": p_kal,
    "ci_low": lo_te,
    "ci_high": hi_te,
    "t2": t2_te,
    "spe": spe_te,
    "kalman_bias": biases,
    "rolling_rmse_static": pd.Series((y_te - p_stat) ** 2).rolling(45, min_periods=10).mean().pipe(np.sqrt).values,
    "rolling_rmse_kalman": pd.Series((y_te - p_kal) ** 2).rolling(45, min_periods=10).mean().pipe(np.sqrt).values,
})

print(f"Paquete entrenado | RMSE Val (Q10/90 CQR={cqr_offset:.2f}C) | RMSE Test Estático={np.sqrt(mean_squared_error(y_te, p_stat)):.3f} -> Kalman={np.sqrt(mean_squared_error(y_te, p_kal)):.3f} °C")

Paquete entrenado | RMSE Val (Q10/90 CQR=0.44C) | RMSE Test Estático=3.941 -> Kalman=1.899 °C


In [2]:
# Serializar bundle completo de producción
bundle = {
    "model_point": model_point,
    "model_q10": model_q10,
    "model_q90": model_q90,
    "cqr_offset": cqr_offset,
    "pca_monitor": {
        "scaler": sc_pca,
        "pca": pca,
        "eigenvalues": pca.explained_variance_,
        "t2_limit_95": float(np.percentile(t2_hist, 95)),
        "t2_limit_99": float(np.percentile(t2_hist, 99)),
        "spe_limit_95": float(np.percentile(spe_hist, 95)),
        "spe_limit_99": float(np.percentile(spe_hist, 99)),
    },
    "kalman_params": {"Q": 0.25, "R": 0.30, "initial_bias": 0.0, "initial_P": 1.0},
    "feature_cols": feature_cols,
    "history_eval": history_eval,
    "version": "2.0.0",
    "metrics": {
        "test_static_rmse": float(np.sqrt(mean_squared_error(y_te, p_stat))),
        "test_kalman_rmse": float(np.sqrt(mean_squared_error(y_te, p_kal))),
        "test_kalman_mae": float(mean_absolute_error(y_te, p_kal)),
        "test_kalman_r2": float(r2_score(y_te, p_kal)),
    },
}

pkl_path = os.path.join(WORK_DIR, "sensor_virtual_v1.pkl")
joblib.dump(bundle, pkl_path)
print(f"Bundle de producción exportado en: {os.path.abspath(pkl_path)}")

Bundle de producción exportado en: /Users/santiagozapata/pungo-academy/course-production/sensores-virtuales/notebooks/sensor_virtual_v1.pkl


In [3]:
# Función de inferencia en tiempo real + pruebas de aceptación en 3 escenarios
explainer_prod = shap.TreeExplainer(bundle["model_point"])

def predecir_flash_point(lectura_dcs: dict, bundle: dict, kalman_bias: float = 0.0) -> dict:
    tg = lectura_dcs["kerosene_draw_temp_C"] - lectura_dcs["column_top_temp_C"]
    row = {
        **lectura_dcs,
        "temp_gradient": tg,
        "tray_to_draw_diff": lectura_dcs["tray_15_temp_C"] - lectura_dcs["kerosene_draw_temp_C"],
        "specific_steam": lectura_dcs["stripping_steam_flow_kgh"] / (lectura_dcs["kerosene_product_flow_m3h"] + 1e-6),
        "yield_ratio": lectura_dcs["kerosene_product_flow_m3h"] / (lectura_dcs["feed_flow_rate_m3h"] + 1e-6),
        "condenser_approach": lectura_dcs["condenser_outlet_temp_C"] - lectura_dcs["ambient_temp_C"],
        "kerosene_draw_temp_lag1": lectura_dcs["kerosene_draw_temp_C"] - 0.3,
        "kerosene_draw_temp_lag2": lectura_dcs["kerosene_draw_temp_C"] - 0.6,
        "kerosene_draw_temp_ma9": lectura_dcs["kerosene_draw_temp_C"],
        "temp_gradient_lag1": tg - 0.2,
        "post_crude_change": 1 if lectura_dcs["crude_api_gravity"] < 31.0 else 0,
    }
    Xr = pd.DataFrame([[row[c] for c in bundle["feature_cols"]]], columns=bundle["feature_cols"])
    pred = float(np.clip(bundle["model_point"].predict(Xr)[0] + kalman_bias, 30.0, 72.0))
    cqr = bundle["cqr_offset"]
    q10 = float(np.clip(bundle["model_q10"].predict(Xr)[0] - cqr + kalman_bias, 28.0, pred))
    q90 = float(np.clip(bundle["model_q90"].predict(Xr)[0] + cqr + kalman_bias, pred, 74.0))

    pm = bundle["pca_monitor"]
    Xs = pm["scaler"].transform(Xr)
    sc = pm["pca"].transform(Xs)
    t2 = float(np.sum((sc ** 2) / pm["eigenvalues"], axis=1)[0])
    spe = float(np.sum((Xs - pm["pca"].inverse_transform(sc)) ** 2, axis=1)[0])
    dom_ok = (t2 <= pm["t2_limit_95"]) and (spe <= pm["spe_limit_95"])

    sv = pd.Series(explainer_prod.shap_values(Xr)[0], index=bundle["feature_cols"])
    top3 = sv.reindex(sv.abs().sort_values(ascending=False).head(3).index).round(2).to_dict()

    if pred < 38.0 or pred > 60.0:
        estado = "FUERA DE ESPECIFICACION"
    elif pred < 40.5 or pred > 57.5:
        estado = "ADVERTENCIA"
    else:
        estado = "DENTRO DE ESPECIFICACION"

    return {
        "flash_point_C": round(pred, 1),
        "ic_80_C": (round(q10, 1), round(q90, 1)),
        "estado_espec": estado,
        "t2": round(t2, 2),
        "spe": round(spe, 2),
        "dominio_valido": dom_ok,
        "top_shap": top3,
    }

base_row = df[df["split"] == "train"].iloc[100].to_dict()
escenarios = {
    "1. Operación Normal (API 33.5)": base_row,
    "2. Límite Bajo (Extracción fría)": {**base_row, "kerosene_draw_temp_C": 171.5, "stripping_steam_flow_kgh": 1850.0, "column_delta_p_kPa": 21.8},
    "3. Crudo Extra-Pesado + Deriva": {**base_row, "crude_api_gravity": 26.2, "kerosene_draw_temp_C": 188.5, "reflux_ratio": 2.26},
}
for nombre, esc in escenarios.items():
    print(f"\n--- {nombre} ---")
    print(predecir_flash_point(esc, bundle))


--- 1. Operación Normal (API 33.5) ---
{'flash_point_C': 49.4, 'ic_80_C': (47.4, 51.4), 'estado_espec': 'DENTRO DE ESPECIFICACION', 't2': 6.25, 'spe': 0.35, 'dominio_valido': True, 'top_shap': {'specific_steam': 1.05, 'crude_api_gravity': -0.21, 'column_delta_p_kPa': 0.21}}

--- 2. Límite Bajo (Extracción fría) ---
{'flash_point_C': 36.4, 'ic_80_C': (34.7, 48.6), 'estado_espec': 'FUERA DE ESPECIFICACION', 't2': 82.94, 'spe': 0.85, 'dominio_valido': False, 'top_shap': {'kerosene_draw_temp_C': -6.89, 'specific_steam': -1.89, 'kerosene_draw_temp_lag1': -1.54}}

--- 3. Crudo Extra-Pesado + Deriva ---
{'flash_point_C': 58.6, 'ic_80_C': (48.6, 60.7), 'estado_espec': 'ADVERTENCIA', 't2': 65.48, 'spe': 6.93, 'dominio_valido': False, 'top_shap': {'kerosene_draw_temp_C': 6.44, 'kerosene_draw_temp_lag1': 1.35, 'specific_steam': 1.0}}


In [4]:
# Escribir la aplicación completa de Sala de Control en Streamlit (main.py)
streamlit_code = '"""\nmain.py — Sensor Virtual: Flash Point del Keroseno (Control Room HMI v2.0)\nAplicacion Streamlit con Intervalos Cuantilicos (P10-P90), Explicabilidad SHAP,\nMonitoreo de Dominio (Hotelling T2 / SPE) y Correccion de Sesgo por Filtro de Kalman.\n\nPungo Tech · www.pungoapp.com · contacto@pungoapp.com · Quito, Ecuador\nUso:\n    streamlit run main.py\n"""\n\nimport os\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport shap\nimport streamlit as st\n\nst.set_page_config(\n    page_title="Sensor Virtual · Flash Point Keroseno | Pungo Tech",\n    layout="wide",\n    initial_sidebar_state="expanded",\n)\n\n# Estilo visual Pungo Tech 2027\nst.markdown(\n    """\n    <style>\n    .pungo-header {\n        background: linear-gradient(135deg, #0D202B 0%, #004E6B 100%);\n        color: #FFFFFF;\n        padding: 1.1rem 1.5rem;\n        border-radius: 12px;\n        border-bottom: 4px solid #8EC14B;\n        margin-bottom: 1rem;\n    }\n    .pungo-header h1 { margin: 0; font-size: 1.65rem; font-weight: 700; color: #FFFFFF; }\n    .pungo-header p { margin: 0.25rem 0 0 0; font-size: 0.92rem; color: #CFE8F3; }\n    .status-card {\n        padding: 0.9rem 1.1rem;\n        border-radius: 10px;\n        border-left: 6px solid;\n        margin-bottom: 0.5rem;\n    }\n    .status-ok   { background: #ECFDF5; border-color: #27AE60; color: #065F46; }\n    .status-warn { background: #FFFBEB; border-color: #E8820C; color: #92400E; }\n    .status-bad  { background: #FEF2F2; border-color: #C0392B; color: #991B1B; }\n    </style>\n    """,\n    unsafe_allow_html=True,\n)\n\nst.markdown(\n    """\n    <div class="pungo-header">\n        <h1>Sensor Virtual en Tiempo Real: Flash Point del Keroseno</h1>\n        <p>Unidad de Destilación Atmosférica · Analizador Inferencial v2.0 (LightGBM + CQR + SHAP + T²/SPE + Kalman) | <b>Pungo Tech</b></p>\n    </div>\n    """,\n    unsafe_allow_html=True,\n)\n\n\n@st.cache_resource\ndef load_bundle():\n    for candidate in ["sensor_virtual_v1.pkl", "/content/sensor_virtual_v1.pkl", "app/sensor_virtual_v1.pkl"]:\n        if os.path.exists(candidate):\n            return joblib.load(candidate)\n    raise FileNotFoundError("No se encontró sensor_virtual_v1.pkl en el directorio actual ni en /content/.")\n\n\n@st.cache_resource\ndef get_explainer(_model):\n    return shap.TreeExplainer(_model)\n\n\nPRESETS = {\n    "Operación Normal (Crudo API 33.5)": {\n        "kerosene_draw_temp_C": 179.5,\n        "column_top_temp_C": 109.5,\n        "tray_15_temp_C": 200.0,\n        "stripping_steam_flow_kgh": 2190.0,\n        "kerosene_product_flow_m3h": 109.0,\n        "reflux_ratio": 2.08,\n        "column_delta_p_kPa": 20.0,\n        "column_top_pressure_kPag": 110.0,\n        "ambient_temp_C": 15.0,\n        "crude_api_gravity": 33.5,\n        "feed_flow_rate_m3h": 955.0,\n        "feed_temperature_C": 364.5,\n        "column_bottom_temp_C": 355.0,\n        "reflux_flow_m3h": 298.0,\n        "condenser_outlet_temp_C": 44.5,\n        "kalman_bias": 0.0,\n    },\n    "Alerta Límite Bajo (< 40 °C)": {\n        "kerosene_draw_temp_C": 171.8,\n        "column_top_temp_C": 111.5,\n        "tray_15_temp_C": 192.5,\n        "stripping_steam_flow_kgh": 1880.0,\n        "kerosene_product_flow_m3h": 112.0,\n        "reflux_ratio": 1.96,\n        "column_delta_p_kPa": 21.8,\n        "column_top_pressure_kPag": 114.5,\n        "ambient_temp_C": 4.0,\n        "crude_api_gravity": 33.2,\n        "feed_flow_rate_m3h": 970.0,\n        "feed_temperature_C": 361.0,\n        "column_bottom_temp_C": 350.0,\n        "reflux_flow_m3h": 286.0,\n        "condenser_outlet_temp_C": 41.2,\n        "kalman_bias": 0.0,\n    },\n    "Cambio de Crudo (API 29.0)": {\n        "kerosene_draw_temp_C": 181.2,\n        "column_top_temp_C": 110.8,\n        "tray_15_temp_C": 203.2,\n        "stripping_steam_flow_kgh": 2290.0,\n        "kerosene_product_flow_m3h": 105.5,\n        "reflux_ratio": 2.14,\n        "column_delta_p_kPa": 20.6,\n        "column_top_pressure_kPag": 111.0,\n        "ambient_temp_C": 22.0,\n        "crude_api_gravity": 29.0,\n        "feed_flow_rate_m3h": 948.0,\n        "feed_temperature_C": 367.0,\n        "column_bottom_temp_C": 360.5,\n        "reflux_flow_m3h": 308.0,\n        "condenser_outlet_temp_C": 46.8,\n        "kalman_bias": -0.6,\n    },\n    "Crudo Extra-Pesado + Deriva (T² Alto)": {\n        "kerosene_draw_temp_C": 188.5,\n        "column_top_temp_C": 112.6,\n        "tray_15_temp_C": 206.8,\n        "stripping_steam_flow_kgh": 2480.0,\n        "kerosene_product_flow_m3h": 99.0,\n        "reflux_ratio": 2.25,\n        "column_delta_p_kPa": 22.4,\n        "column_top_pressure_kPag": 115.5,\n        "ambient_temp_C": 28.0,\n        "crude_api_gravity": 26.2,\n        "feed_flow_rate_m3h": 932.0,\n        "feed_temperature_C": 370.5,\n        "column_bottom_temp_C": 366.0,\n        "reflux_flow_m3h": 324.0,\n        "condenser_outlet_temp_C": 49.5,\n        "kalman_bias": -2.8,\n    },\n}\n\n\ndef init_state():\n    defaults = PRESETS["Operación Normal (Crudo API 33.5)"]\n    for k, v in defaults.items():\n        if k not in st.session_state:\n            st.session_state[k] = v\n\n\ndef apply_preset(name: str):\n    for k, v in PRESETS[name].items():\n        st.session_state[k] = v\n\n\ndef build_feature_row(vals: dict, feature_cols: list[str]) -> pd.DataFrame:\n    temp_grad = vals["kerosene_draw_temp_C"] - vals["column_top_temp_C"]\n    tray_diff = vals["tray_15_temp_C"] - vals["kerosene_draw_temp_C"]\n    spec_steam = vals["stripping_steam_flow_kgh"] / (vals["kerosene_product_flow_m3h"] + 1e-6)\n    yield_ratio = vals["kerosene_product_flow_m3h"] / (vals["feed_flow_rate_m3h"] + 1e-6)\n    cond_approach = vals["condenser_outlet_temp_C"] - vals["ambient_temp_C"]\n\n    row = {\n        **vals,\n        "temp_gradient": temp_grad,\n        "tray_to_draw_diff": tray_diff,\n        "specific_steam": spec_steam,\n        "yield_ratio": yield_ratio,\n        "condenser_approach": cond_approach,\n        "kerosene_draw_temp_lag1": vals["kerosene_draw_temp_C"] - 0.3,\n        "kerosene_draw_temp_lag2": vals["kerosene_draw_temp_C"] - 0.6,\n        "kerosene_draw_temp_ma9": vals["kerosene_draw_temp_C"],\n        "temp_gradient_lag1": temp_grad - 0.2,\n        "post_crude_change": 1 if vals["crude_api_gravity"] < 31.0 else 0,\n    }\n    return pd.DataFrame([[row.get(c, 0.0) for c in feature_cols]], columns=feature_cols)\n\n\ndef predict_full(bundle: dict, vals: dict, kalman_bias: float):\n    X_row = build_feature_row(vals, bundle["feature_cols"])\n    pred_raw = float(bundle["model_point"].predict(X_row)[0])\n    pred = float(np.clip(pred_raw + kalman_bias, 30.0, 72.0))\n\n    cqr = float(bundle.get("cqr_offset", 0.0))\n    q10 = float(np.clip(bundle["model_q10"].predict(X_row)[0] - cqr + kalman_bias, 28.0, pred))\n    q90 = float(np.clip(bundle["model_q90"].predict(X_row)[0] + cqr + kalman_bias, pred, 74.0))\n\n    # PCA Hotelling T2 & SPE domain monitoring\n    pca_mon = bundle["pca_monitor"]\n    X_s = pca_mon["scaler"].transform(X_row)\n    scores = pca_mon["pca"].transform(X_s)\n    t2_val = float(np.sum((scores ** 2) / pca_mon["eigenvalues"], axis=1)[0])\n    rec = pca_mon["pca"].inverse_transform(scores)\n    spe_val = float(np.sum((X_s - rec) ** 2, axis=1)[0])\n\n    # SHAP explanation\n    explainer = get_explainer(bundle["model_point"])\n    sv = explainer.shap_values(X_row)[0]\n    base_val = float(np.atleast_1d(explainer.expected_value)[0])\n    shap_series = pd.Series(sv, index=bundle["feature_cols"])\n\n    return {\n        "X_row": X_row,\n        "pred": pred,\n        "pred_raw": pred_raw,\n        "q10": q10,\n        "q90": q90,\n        "t2": t2_val,\n        "t2_lim95": float(pca_mon["t2_limit_95"]),\n        "t2_lim99": float(pca_mon["t2_limit_99"]),\n        "spe": spe_val,\n        "spe_lim95": float(pca_mon["spe_limit_95"]),\n        "spe_lim99": float(pca_mon["spe_limit_99"]),\n        "shap": shap_series,\n        "shap_base": base_val,\n    }\n\n\ntry:\n    bundle = load_bundle()\n    init_state()\n    bundle_ok = True\nexcept Exception as exc:\n    st.error(f"Error cargando modelo: {exc}")\n    bundle_ok = False\n\nif bundle_ok:\n    st.sidebar.header("Simulador de Sala de Control")\n    st.sidebar.markdown("**Escenarios Operativos Rápidos**")\n    preset_choice = st.sidebar.selectbox(\n        "Cargar escenario predefinido",\n        list(PRESETS.keys()),\n        index=0,\n    )\n    if st.sidebar.button("Aplicar Escenario Seleccionado", use_container_width=True):\n        apply_preset(preset_choice)\n        st.rerun()\n\n    st.sidebar.divider()\n    st.sidebar.subheader("Variables Clave del DCS")\n    st.sidebar.slider("Temp. extracción keroseno (°C)", 165.0, 205.0, step=0.2, key="kerosene_draw_temp_C")\n    st.sidebar.slider("Temp. tope de torre (°C)", 103.0, 118.0, step=0.2, key="column_top_temp_C")\n    st.sidebar.slider("Temp. Plato 15 (°C)", 182.0, 222.0, step=0.5, key="tray_15_temp_C")\n    st.sidebar.slider("Vapor de despojamiento (kg/h)", 1650.0, 2750.0, step=10.0, key="stripping_steam_flow_kgh")\n    st.sidebar.slider("Flujo producto keroseno (m³/h)", 85.0, 135.0, step=0.5, key="kerosene_product_flow_m3h")\n    st.sidebar.slider("Relación de reflujo", 1.85, 2.35, step=0.01, key="reflux_ratio")\n    st.sidebar.slider("Caída de presión ΔP (kPa)", 17.0, 24.5, step=0.1, key="column_delta_p_kPa")\n    st.sidebar.slider("Presión tope torre (kPag)", 102.0, 122.0, step=0.5, key="column_top_pressure_kPag")\n    st.sidebar.slider("Temperatura ambiente (°C)", -5.0, 35.0, step=0.5, key="ambient_temp_C")\n    st.sidebar.slider("Gravedad API del crudo", 24.5, 36.0, step=0.1, key="crude_api_gravity")\n\n    with st.sidebar.expander("Variables Secundarias del DCS"):\n        st.slider("Caudal alimentación (m³/h)", 880.0, 1040.0, step=2.0, key="feed_flow_rate_m3h")\n        st.slider("Temp. alimentación (°C)", 352.0, 378.0, step=0.5, key="feed_temperature_C")\n        st.slider("Temp. fondo de torre (°C)", 342.0, 372.0, step=0.5, key="column_bottom_temp_C")\n        st.slider("Flujo de reflujo (m³/h)", 260.0, 350.0, step=1.0, key="reflux_flow_m3h")\n        st.slider("Temp. salida condensador (°C)", 35.0, 56.0, step=0.5, key="condenser_outlet_temp_C")\n\n    st.sidebar.divider()\n    st.sidebar.subheader("Filtro de Kalman (Lab Bias)")\n    kalman_bias = st.sidebar.slider(\n        "Corrección de sesgo Kalman (°C)",\n        -4.5,\n        4.5,\n        step=0.1,\n        key="kalman_bias",\n        help="Offset recursivo estimado contra las últimas muestras de laboratorio cada 8 h.",\n    )\n\n    current_vals = {c: float(st.session_state[c]) for c in PRESETS["Operación Normal (Crudo API 33.5)"] if c != "kalman_bias"}\n    out = predict_full(bundle, current_vals, kalman_bias)\n\n    pred = out["pred"]\n    q10, q90 = out["q10"], out["q90"]\n    t2, t2_lim = out["t2"], out["t2_lim95"]\n    spe, spe_lim = out["spe"], out["spe_lim95"]\n    domain_ok = (t2 <= t2_lim) and (spe <= spe_lim)\n\n    # Clasificación operativa de especificación (38 - 60 C)\n    if pred < 38.0:\n        spec_css, spec_label = "status-bad", "FUERA DE ESPECIFICACIÓN (BAJO < 38 °C)"\n        action_msg = "Riesgo de seguridad: subir temperatura de extracción o vapor de despojamiento; retener envío a tanque."\n    elif pred > 60.0:\n        spec_css, spec_label = "status-bad", "FUERA DE ESPECIFICACIÓN (ALTO > 60 °C)"\n        action_msg = "Fracción pesada excesiva: bajar temperatura de extracción o incrementar reflujo superior."\n    elif pred < 40.5 or pred > 57.5:\n        spec_css, spec_label = "status-warn", "ADVERTENCIA — CERCA DEL LÍMITE"\n        action_msg = "Monitoreo estrecho: ajustar ligeramente punto de corte para recuperar margen hacia 48 °C."\n    else:\n        spec_css, spec_label = "status-ok", "DENTRO DE ESPECIFICACIÓN (38–60 °C)"\n        action_msg = "Operación nominal estable. Mantener lazo de control actual."\n\n    # Fila 1: KPI Cards\n    k1, k2, k3, k4 = st.columns([1.2, 1.5, 1.3, 1.1])\n    with k1:\n        st.metric(\n            "Flash Point Predicho",\n            f"{pred:.1f} °C",\n            delta=f"{pred - 48.0:+.1f} °C vs objetivo (48 °C)",\n        )\n        st.caption(f"Intervalo 80% (CQR): **[{q10:.1f} – {q90:.1f} °C]**")\n\n    with k2:\n        st.markdown(\n            f\'<div class="status-card {spec_css}"><b>{spec_label}</b><br><span style="font-size:0.86rem;">{action_msg}</span></div>\',\n            unsafe_allow_html=True,\n        )\n\n    with k3:\n        if domain_ok:\n            dom_css = "status-ok"\n            dom_title = "DOMINIO VÁLIDO (T² y SPE OK)"\n            dom_sub = f"T² = {t2:.1f} (lím {t2_lim:.1f}) · SPE = {spe:.2f} (lím {spe_lim:.2f})"\n        else:\n            dom_css = "status-warn" if (t2 <= out["t2_lim99"] and spe <= out["spe_lim99"]) else "status-bad"\n            dom_title = "ALERTA DE DOMINIO (EXTRAPOLACIÓN / DERIVA)"\n            dom_sub = f"T² = {t2:.1f} (lím {t2_lim:.1f}) · SPE = {spe:.2f} (lím {spe_lim:.2f})"\n        st.markdown(\n            f\'<div class="status-card {dom_css}"><b>{dom_title}</b><br><span style="font-size:0.84rem;">{dom_sub}</span></div>\',\n            unsafe_allow_html=True,\n        )\n\n    with k4:\n        marg_inf = pred - 38.0\n        marg_sup = 60.0 - pred\n        st.metric("Margen Inferior (38 °C)", f"{marg_inf:+.1f} °C")\n        st.caption(f"Margen sup: **{marg_sup:+.1f} °C** · Bias Kalman: **{kalman_bias:+.1f} °C**")\n\n    st.divider()\n\n    # Fila 2: Gauge Operativo + Explicabilidad SHAP en Vivo\n    col_gauge, col_shap = st.columns([1.05, 1.15])\n\n    with col_gauge:\n        st.subheader("Indicador Operativo y Banda de Incertidumbre (80%)")\n        fig_g, ax_g = plt.subplots(figsize=(7.2, 3.2), dpi=150)\n        # Bandas de especificación\n        ax_g.axvspan(30, 38, color="#FADBD8", alpha=0.85, label="Fuera espec. (<38 °C)")\n        ax_g.axvspan(38, 40.5, color="#FCF3CF", alpha=0.90, label="Zona advertencia")\n        ax_g.axvspan(40.5, 57.5, color="#D5F5E3", alpha=0.85, label="Zona óptima (38–60 °C)")\n        ax_g.axvspan(57.5, 60, color="#FCF3CF", alpha=0.90)\n        ax_g.axvspan(60, 72, color="#FADBD8", alpha=0.85)\n        ax_g.axvline(38, color="#C0392B", linestyle="--", linewidth=1.6)\n        ax_g.axvline(48, color="#007297", linestyle=":", linewidth=1.8, label="Objetivo (48.0 °C)")\n        ax_g.axvline(60, color="#C0392B", linestyle="--", linewidth=1.6)\n\n        # Banda P10-P90 + marcador puntual\n        ax_g.hlines(0.5, q10, q90, color="#0D202B", linewidth=10, alpha=0.35, label=f"IC 80% [{q10:.1f}–{q90:.1f} °C]")\n        ax_g.scatter([pred], [0.5], color="#0D202B", s=260, zorder=5, edgecolor="#FFFFFF", linewidth=2)\n        ax_g.text(\n            pred, 0.70, f"{pred:.1f} °C", ha="center", va="bottom",\n            fontsize=14, fontweight="bold", color="#0D202B",\n            bbox=dict(boxstyle="round,pad=0.25", facecolor="#FFFFFF", edgecolor="#0D202B", linewidth=1.2, alpha=0.95),\n            zorder=6,\n        )\n        ax_g.text(\n            (q10 + q90) / 2, 0.25, f"IC 80%: {q10:.1f} – {q90:.1f} °C",\n            ha="center", va="top", fontsize=9.5, color="#0D202B", fontweight="bold",\n            bbox=dict(boxstyle="round,pad=0.2", facecolor="#FFFFFF", edgecolor="#CBD5E1", alpha=0.92),\n            zorder=6,\n        )\n\n        ax_g.set_xlim(30, 72)\n        ax_g.set_ylim(0, 1)\n        ax_g.set_yticks([])\n        ax_g.set_xlabel("Flash Point (°C)", fontsize=10, fontweight="bold")\n        ax_g.legend(loc="upper right", fontsize=7.5, ncol=2, framealpha=0.92)\n        plt.tight_layout()\n        st.pyplot(fig_g)\n        plt.close(fig_g)\n\n    with col_shap:\n        st.subheader("Explicabilidad SHAP en Vivo (¿Qué impulsa la predicción?)")\n        top_sv = out["shap"].reindex(out["shap"].abs().sort_values(ascending=False).head(6).index)\n        fig_s, ax_s = plt.subplots(figsize=(7.2, 3.2), dpi=150)\n        vals_s = top_sv.values[::-1]\n        names_s = top_sv.index[::-1]\n        bar_colors = ["#007297" if v >= 0 else "#E8820C" for v in vals_s]\n        bars = ax_s.barh(names_s, vals_s, color=bar_colors, edgecolor="white", height=0.62)\n        ax_s.axvline(0, color="#0D202B", linewidth=1.0)\n        for b, val in zip(bars, vals_s):\n            offset = 0.06 if val >= 0 else -0.06\n            ha = "left" if val >= 0 else "right"\n            ax_s.text(val + offset, b.get_y() + b.get_height() / 2, f"{val:+.2f} °C", va="center", ha=ha, fontsize=8.5, fontweight="bold")\n        max_abs = max(1.2, float(np.max(np.abs(vals_s))) * 1.28)\n        ax_s.set_xlim(-max_abs, max_abs)\n        ax_s.set_xlabel("Contribución marginal SHAP (°C respecto al valor base)", fontsize=9.5)\n        ax_s.set_title(f"Base = {out[\'shap_base\']:.1f} °C → Predicción sin Kalman = {out[\'pred_raw\']:.1f} °C", fontsize=10)\n        plt.tight_layout()\n        st.pyplot(fig_s)\n        plt.close(fig_s)\n\n    # Fila 3: Pestañas Analíticas\n    tab1, tab2, tab3 = st.tabs([\n        "Monitor de Deriva y Filtro de Kalman",\n        "Dominio de Aplicabilidad (Hotelling T² y SPE)",\n        "Simulador What-If (Sensibilidad)",\n    ])\n\n    hist_df = bundle.get("history_eval")\n    with tab1:\n        if hist_df is not None:\n            fig_d, axes = plt.subplots(2, 1, figsize=(12, 5.6), dpi=150, sharex=True)\n            x_dates = pd.to_datetime(hist_df["date"])\n            axes[0].fill_between(x_dates, hist_df["ci_low"], hist_df["ci_high"], color="#007297", alpha=0.14, label="Banda CQR 80%")\n            axes[0].plot(x_dates, hist_df["y_true"], color="#0D202B", linewidth=1.4, label="Laboratorio (Real)")\n            axes[0].plot(x_dates, hist_df["y_pred_static"], color="#C0392B", linestyle="--", linewidth=1.2, alpha=0.85, label="Modelo Estático (deriva por termopozo)")\n            axes[0].plot(x_dates, hist_df["y_pred_kalman"], color="#8EC14B", linewidth=1.8, label="LightGBM + Filtro de Kalman")\n            axes[0].axhline(38, color="#C0392B", linestyle=":", linewidth=1.0)\n            axes[0].axhline(60, color="#C0392B", linestyle=":", linewidth=1.0)\n            axes[0].set_ylabel("Flash Point (°C)")\n            axes[0].set_title("Evaluación en Producción (Q4): Corrección de Deriva +4.2 °C mediante Filtro de Kalman", fontweight="bold")\n            axes[0].legend(loc="upper left", fontsize=8.5, ncol=2)\n\n            axes[1].plot(x_dates, hist_df["rolling_rmse_static"], color="#C0392B", linewidth=1.5, label="RMSE deslizante Estático (45 turnos)")\n            axes[1].plot(x_dates, hist_df["rolling_rmse_kalman"], color="#007297", linewidth=1.8, label="RMSE deslizante + Kalman")\n            axes[1].axhline(2.2, color="#E8820C", linestyle="--", linewidth=1.2, label="Umbral de alerta (2.2 °C)")\n            axes[1].set_ylabel("RMSE (°C)")\n            axes[1].set_xlabel("Fecha")\n            axes[1].legend(loc="upper left", fontsize=8.5)\n            plt.tight_layout()\n            st.pyplot(fig_d)\n            plt.close(fig_d)\n        else:\n            st.info("No hay histórico de evaluación adjunto en el archivo del modelo.")\n\n    with tab2:\n        if hist_df is not None:\n            fig_t, axes = plt.subplots(2, 1, figsize=(12, 5.2), dpi=150, sharex=True)\n            x_dates = pd.to_datetime(hist_df["date"])\n            axes[0].plot(x_dates, hist_df["t2"], color="#007297", linewidth=1.4, label="Hotelling T²")\n            axes[0].axhline(t2_lim, color="#E8820C", linestyle="--", linewidth=1.3, label=f"Límite 95% ({t2_lim:.1f})")\n            axes[0].axhline(out["t2_lim99"], color="#C0392B", linestyle=":", linewidth=1.3, label=f"Límite 99% ({out[\'t2_lim99\']:.1f})")\n            axes[0].set_ylabel("T²")\n            axes[0].set_title("Monitoreo Multivariado PCA: Detección Temprana de Cambio de Crudo y Deriva", fontweight="bold")\n            axes[0].legend(loc="upper left", fontsize=8.5)\n\n            axes[1].plot(x_dates, hist_df["spe"], color="#8EC14B", linewidth=1.4, label="SPE (Q-statistic)")\n            axes[1].axhline(spe_lim, color="#E8820C", linestyle="--", linewidth=1.3, label=f"Límite 95% ({spe_lim:.2f})")\n            axes[1].axhline(out["spe_lim99"], color="#C0392B", linestyle=":", linewidth=1.3, label=f"Límite 99% ({out[\'spe_lim99\']:.2f})")\n            axes[1].set_ylabel("SPE")\n            axes[1].set_xlabel("Fecha")\n            axes[1].legend(loc="upper left", fontsize=8.5)\n            plt.tight_layout()\n            st.pyplot(fig_t)\n            plt.close(fig_t)\n\n    with tab3:\n        st.markdown("Simula el impacto de ajustar la **Temperatura de Extracción del Keroseno** (`±8 °C`) manteniendo el resto de la torre constante:")\n        sweep_temps = np.linspace(max(165.0, current_vals["kerosene_draw_temp_C"] - 8.0), min(205.0, current_vals["kerosene_draw_temp_C"] + 8.0), 35)\n        preds_w, lo_w, hi_w = [], [], []\n        cqr = float(bundle.get("cqr_offset", 0.0))\n        for td in sweep_temps:\n            v_copy = dict(current_vals)\n            v_copy["kerosene_draw_temp_C"] = float(td)\n            Xr = build_feature_row(v_copy, bundle["feature_cols"])\n            p = float(np.clip(bundle["model_point"].predict(Xr)[0] + kalman_bias, 30.0, 72.0))\n            preds_w.append(p)\n            lo_w.append(float(np.clip(bundle["model_q10"].predict(Xr)[0] - cqr + kalman_bias, 28.0, p)))\n            hi_w.append(float(np.clip(bundle["model_q90"].predict(Xr)[0] + cqr + kalman_bias, p, 74.0)))\n\n        fig_w, ax_w = plt.subplots(figsize=(11, 3.8), dpi=150)\n        ax_w.fill_between(sweep_temps, lo_w, hi_w, color="#007297", alpha=0.18, label="Intervalo CQR 80%")\n        ax_w.plot(sweep_temps, preds_w, color="#0D202B", linewidth=2.2, label="Flash Point predicho")\n        ax_w.scatter([current_vals["kerosene_draw_temp_C"]], [pred], color="#8EC14B", s=110, zorder=5, edgecolor="#0D202B", label="Punto actual")\n        ax_w.axhline(38.0, color="#C0392B", linestyle="--", label="Límite inf (38 °C)")\n        ax_w.axhline(48.0, color="#007297", linestyle=":", label="Objetivo (48 °C)")\n        ax_w.axhline(60.0, color="#C0392B", linestyle="--", label="Límite sup (60 °C)")\n        ax_w.set_xlabel("Temp. extracción keroseno (°C)")\n        ax_w.set_ylabel("Flash Point predicho (°C)")\n        ax_w.set_title("Curva What-If de Sensibilidad Operativa", fontweight="bold")\n        ax_w.legend(loc="upper left", fontsize=8.5, ncol=3)\n        plt.tight_layout()\n        st.pyplot(fig_w)\n        plt.close(fig_w)\n\n    st.divider()\n    st.caption(\n        f"Sensor Virtual v{bundle.get(\'version\', \'2.0.0\')} | LightGBM + CQR + SHAP + Hotelling T²/SPE + Kalman | "\n        "Pungo Tech · www.pungoapp.com · contacto@pungoapp.com · Quito, Ecuador"\n    )\n'

main_py_path = os.path.join(WORK_DIR, "main.py")
with open(main_py_path, "w", encoding="utf-8") as f:
    f.write(streamlit_code)

print(f"Aplicación Streamlit escrita en: {os.path.abspath(main_py_path)} ({len(streamlit_code.splitlines())} líneas)")
print("Para ejecutar localmente: streamlit run main.py")

Aplicación Streamlit escrita en: /Users/santiagozapata/pungo-academy/course-production/sensores-virtuales/notebooks/main.py (490 líneas)
Para ejecutar localmente: streamlit run main.py


### Checkpoint: Lo que debes ver

- `sensor_virtual_v1.pkl` exportado con el modelo puntual, cuantiles CQR, monitor PCA ($T^2$/SPE), parámetros Kalman e histórico Q4
- Prueba de aceptación exitosa sobre 3 escenarios industriales (Normal, Alerta Límite Bajo y Extrapolación $T^2$ fuera de dominio)
- Archivo `main.py` generado y listo para ejecutarse con `streamlit run main.py`